# Chapter 5: Dense Map表現 — Occupancy Grid, SDF, TSDF

**SLAM Handbook Chapter 5** — 密な地図表現の基礎。

## このNotebookで学ぶこと

1. **Occupancy Grid Map** — log-oddsによるベイズ更新、レイキャスティング
2. **Signed Distance Field (SDF)** — 陰的表面表現、ゼロ交差が表面
3. **Truncated SDF (TSDF)** — KinectFusion風の深度統合
4. **表現の比較** — Point Cloud, Surfel, Voxel, Mesh の特徴

### 対応セクション
- Section 5.1: レンジセンシングの基礎
- Section 5.2: Occupancy Map, Implicit Surface, Distance Field
- Section 5.3: 空間抽象化の種類（Point, Surfel, Voxel, Mesh）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 5.1 2D Occupancy Grid Map

**SLAM Handbook Section 5.2.1, Eq. 5.6-5.7**: 各セル $m_k$ の占有確率をlog-odds形式でベイズ更新。

$$l(m_k | \mathbf{z}_{1:t}) = l(m_k | \mathbf{z}_{1:t-1}) + l(m_k | z_t)$$

ここで $l(\cdot|\cdot) = \log\frac{p(\cdot|\cdot)}{1 - p(\cdot|\cdot)}$ がlog-odds。加算だけで更新できるのが利点。

In [ ]:
# =============================================================
# 2D Occupancy Grid Map の実装
# =============================================================

class OccupancyGrid:
    """2D Occupancy Grid Map (log-odds)"""
    
    def __init__(self, xlim, ylim, resolution):
        self.resolution = resolution
        self.xlim = xlim
        self.ylim = ylim
        self.nx = int((xlim[1] - xlim[0]) / resolution)
        self.ny = int((ylim[1] - ylim[0]) / resolution)
        self.log_odds = np.zeros((self.ny, self.nx))  # log-odds map
        
        # センサモデルのパラメータ
        self.l_occ = 0.85    # 障害物セルのlog-odds更新量
        self.l_free = -0.4   # 自由セルのlog-odds更新量
        self.l_max = 5.0     # クランプ上限
        self.l_min = -5.0    # クランプ下限
    
    def world_to_grid(self, x, y):
        gx = int((x - self.xlim[0]) / self.resolution)
        gy = int((y - self.ylim[0]) / self.resolution)
        return np.clip(gx, 0, self.nx-1), np.clip(gy, 0, self.ny-1)
    
    def grid_to_world(self, gx, gy):
        return self.xlim[0] + (gx + 0.5) * self.resolution, \
               self.ylim[0] + (gy + 0.5) * self.resolution
    
    def update(self, robot_x, robot_y, scan_angles, scan_ranges, max_range=10.0):
        """レーザスキャンでマップを更新 (レイキャスティング)"""
        for angle, r in zip(scan_angles, scan_ranges):
            if r >= max_range:
                r_use = max_range
                hit = False
            else:
                r_use = r
                hit = True
            
            # レイに沿ってBresenham的にセルをたどる
            end_x = robot_x + r_use * np.cos(angle)
            end_y = robot_y + r_use * np.sin(angle)
            
            # レイ上のセルを列挙
            n_steps = int(r_use / self.resolution * 1.5)
            for s in range(n_steps):
                t = s / max(n_steps, 1)
                px = robot_x + t * (end_x - robot_x)
                py = robot_y + t * (end_y - robot_y)
                gx, gy = self.world_to_grid(px, py)
                if 0 <= gx < self.nx and 0 <= gy < self.ny:
                    self.log_odds[gy, gx] += self.l_free
            
            # ヒット点を占有セルとしてマーク
            if hit:
                gx, gy = self.world_to_grid(end_x, end_y)
                if 0 <= gx < self.nx and 0 <= gy < self.ny:
                    self.log_odds[gy, gx] += self.l_occ
        
        # クランプ
        self.log_odds = np.clip(self.log_odds, self.l_min, self.l_max)
    
    def get_probability(self):
        """log-oddsから占有確率に変換"""
        return 1.0 - 1.0 / (1.0 + np.exp(self.log_odds))

# テスト環境: 長方形の部屋 + 障害物
np.random.seed(42)

grid = OccupancyGrid(xlim=(-1, 11), ylim=(-1, 11), resolution=0.1)

# 壁と障害物の定義 (障害物のエッジポイント)
walls = [
    (np.linspace(0, 10, 50), np.zeros(50)),       # 下壁
    (np.linspace(0, 10, 50), np.full(50, 10)),     # 上壁
    (np.zeros(50), np.linspace(0, 10, 50)),        # 左壁
    (np.full(50, 10), np.linspace(0, 10, 50)),     # 右壁
    (np.linspace(3, 3, 20), np.linspace(2, 5, 20)),  # 障害物1 (縦壁)
    (np.linspace(6, 8, 20), np.linspace(7, 7, 20)),  # 障害物2 (横壁)
]

def simulate_scan(robot_x, robot_y, walls, n_beams=72, max_range=10.0):
    """2Dレーザスキャンをシミュレート"""
    angles = np.linspace(0, 2*np.pi, n_beams, endpoint=False)
    ranges = np.full(n_beams, max_range)
    
    for i, angle in enumerate(angles):
        dx, dy = np.cos(angle), np.sin(angle)
        for wall_x, wall_y in walls:
            for wx, wy in zip(wall_x, wall_y):
                # ロボットから壁点への距離
                d = np.sqrt((wx - robot_x)**2 + (wy - robot_y)**2)
                # 壁点がビーム方向に近いか
                bx, by = robot_x + d*dx, robot_y + d*dy
                if np.sqrt((bx-wx)**2 + (by-wy)**2) < 0.15:
                    ranges[i] = min(ranges[i], d)
    
    return angles, ranges

# ロボットの軌跡（4箇所からスキャン）
robot_positions = [(2, 2), (5, 3), (8, 5), (5, 8)]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for rx, ry in robot_positions:
    angles, ranges = simulate_scan(rx, ry, walls)
    grid.update(rx, ry, angles, ranges)

# 占有確率を可視化
prob = grid.get_probability()
axes[0].imshow(prob, origin='lower', cmap='gray_r',
               extent=[grid.xlim[0], grid.xlim[1], grid.ylim[0], grid.ylim[1]])
for rx, ry in robot_positions:
    axes[0].plot(rx, ry, 'r*', markersize=15)
axes[0].set_title('Occupancy Grid Map\n(黒=占有, 白=自由, 灰=未知)', fontweight='bold')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# log-oddsを可視化
im = axes[1].imshow(grid.log_odds, origin='lower', cmap='RdBu_r',
                     extent=[grid.xlim[0], grid.xlim[1], grid.ylim[0], grid.ylim[1]],
                     norm=TwoSlopeNorm(0))
plt.colorbar(im, ax=axes[1], label='Log-odds')
for rx, ry in robot_positions:
    axes[1].plot(rx, ry, 'k*', markersize=15)
axes[1].set_title('Log-odds Map\n(赤=占有, 青=自由)', fontweight='bold')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')

plt.tight_layout()
plt.show()

## 5.2 Signed Distance Field (SDF)

**SLAM Handbook Section 5.2.2, Fig. 5.6**: 各点の値が最寄りの表面までの **符号付き距離**。

- $\text{SDF}(\mathbf{p}) < 0$: 物体の内部
- $\text{SDF}(\mathbf{p}) = 0$: 表面上（ゼロ交差）
- $\text{SDF}(\mathbf{p}) > 0$: 物体の外部

**TSDF** (Truncated SDF): SDFを表面付近のみに制限（$|\text{SDF}| \leq \tau$）。KinectFusion等で使用。

In [ ]:
# =============================================================
# 2D SDF/TSDF の実装と可視化
# =============================================================

# 2Dシーン: 円形障害物
def circle_sdf(x, y, cx, cy, r):
    return np.sqrt((x - cx)**2 + (y - cy)**2) - r

# 複数障害物のSDF (min で合成)
def scene_sdf(x, y):
    d1 = circle_sdf(x, y, 3, 3, 1.5)    # 大きな円
    d2 = circle_sdf(x, y, 7, 6, 1.0)    # 小さな円
    d3 = circle_sdf(x, y, 5, 8, 0.8)    # 小さな円
    return np.minimum(np.minimum(d1, d2), d3)

# グリッド上でSDF計算
resolution = 0.05
x = np.arange(-1, 11, resolution)
y = np.arange(-1, 11, resolution)
X, Y = np.meshgrid(x, y)
SDF = scene_sdf(X, Y)

# TSDF (truncation距離 τ)
tau = 1.5
TSDF = np.clip(SDF, -tau, tau) / tau  # [-1, 1] に正規化

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# SDF
im0 = axes[0].imshow(SDF, origin='lower', cmap='RdBu',
                       extent=[-1, 11, -1, 11], norm=TwoSlopeNorm(0))
axes[0].contour(X, Y, SDF, levels=[0], colors='black', linewidths=2)
plt.colorbar(im0, ax=axes[0], label='SDF value')
axes[0].set_title('Signed Distance Field\n(青=外, 赤=内, 黒線=表面)', fontweight='bold')

# TSDF
im1 = axes[1].imshow(TSDF, origin='lower', cmap='RdBu',
                       extent=[-1, 11, -1, 11], norm=TwoSlopeNorm(0))
axes[1].contour(X, Y, TSDF, levels=[0], colors='black', linewidths=2)
plt.colorbar(im1, ax=axes[1], label='TSDF value')
axes[1].set_title(f'Truncated SDF (τ={tau})\n表面付近のみ情報あり', fontweight='bold')

# SDF勾配（表面法線）
grad_y, grad_x = np.gradient(SDF, resolution)
# 表面付近のみ表示
mask = np.abs(SDF) < 1.0
step = 10
axes[2].imshow(SDF, origin='lower', cmap='RdBu', extent=[-1, 11, -1, 11],
               norm=TwoSlopeNorm(0), alpha=0.3)
axes[2].contour(X, Y, SDF, levels=[0], colors='black', linewidths=2)
axes[2].quiver(X[::step, ::step], Y[::step, ::step],
               grad_x[::step, ::step], grad_y[::step, ::step],
               color='green', alpha=0.6, scale=30)
axes[2].set_title('SDF Gradient (表面法線)\n経路計画に利用', fontweight='bold')

for ax in axes:
    ax.set_aspect('equal')
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.show()

print("→ SDF: 任意の点から最寄表面までの距離を提供")
print("→ TSDF: 表面近傍のみ保持してメモリ節約（KinectFusion等）")
print("→ SDF勾配: 障害物から遠ざかる方向 → motion planningに有用")

## 5.3 演習問題

### 演習1: 動的更新
ロボットが移動しながらスキャンするシミュレーションを作り、Occupancy Gridが逐次更新される様子をアニメーションで可視化してください。

### 演習2: TSDF統合
複数視点からの深度画像をTSDF統合してみましょう。各視点のTSDFを重み付き平均で統合します（KinectFusion方式）。

### 演習3: Occupancy → SDF変換
完成したOccupancy Grid MapからEuclidean Distance Transform (EDT) でSDFを計算してみましょう。`scipy.ndimage.distance_transform_edt` が使えます。

## まとめ

| 表現 | 推定量 | 利点 | 欠点 |
|---|---|---|---|
| **Occupancy Grid** | 占有確率 | シンプル、ベイズ更新が容易 | セル独立仮定、固定解像度 |
| **SDF** | 符号付き距離 | 滑らか、微分可能、衝突判定に便利 | 既知表面が必要 |
| **TSDF** | Truncated SDF | メモリ効率良い、深度統合が容易 | 表面付近のみ |
| **Point Cloud** | 点の集合 | 最もシンプル | 表面情報なし |
| **Surfel** | 点+法線+半径 | メモリ効率 | 接続性なし |
| **Mesh** | 頂点+面 | レンダリング向き、コンパクト | 更新が複雑 |
| **Voxel Grid** | ボクセル値 | 近傍クエリ高速 | メモリ大 |

### 次のNotebook
→ Ch06: Certifiably Optimal Solvers (SDP緩和、SE-Sync)